In [ ]:
import asyncio
import json
import subprocess
import urllib.request
from pathlib import Path


In [ ]:
SRC_DIR = Path.cwd().resolve()
PROJECT_DIR = SRC_DIR.parent
VPN_SCRIPT_PATH = PROJECT_DIR / "scripts" / "nordvpn_connect_country.applescript"
VPN_COUNTRY_SEQUENCE = [
    "Japan",
    "Singapore",
    "United States",
    "Canada",
    "Germany",
    "United Kingdom",
    "Netherlands",
    "France",
    "Sweden",
    "Australia",
    "Switzerland",
    "Spain",
]
VPN_IP_CHANGE_TIMEOUT_SECONDS = 45
VPN_IP_POLL_INTERVAL_SECONDS = 2
vpn_country_index = 0

print(f"VPN 스크립트 경로: {VPN_SCRIPT_PATH}")
print(f"VPN 국가 순서: {VPN_COUNTRY_SEQUENCE}")


In [ ]:
def get_next_vpn_country(vpn_country_index: int) -> tuple[str, int]:
    target_country = VPN_COUNTRY_SEQUENCE[vpn_country_index % len(VPN_COUNTRY_SEQUENCE)]
    return target_country, vpn_country_index + 1


async def switch_vpn_server(country: str) -> dict[str, object]:
    completed = await asyncio.to_thread(
        subprocess.run,
        ["osascript", str(VPN_SCRIPT_PATH), country],
        capture_output=True,
        text=True,
    )
    return {
        "success": completed.returncode == 0,
        "country": country,
        "returncode": completed.returncode,
        "stdout": completed.stdout.strip(),
        "stderr": completed.stderr.strip(),
    }


async def fetch_public_ip() -> str | None:
    def _fetch() -> str | None:
        try:
            with urllib.request.urlopen("https://api.ipify.org?format=json", timeout=10) as response:
                payload = json.loads(response.read().decode("utf-8"))
                return payload.get("ip")
        except Exception:
            return None

    return await asyncio.to_thread(_fetch)


async def wait_for_public_ip_change(previous_ip: str | None) -> dict[str, object]:
    if previous_ip is None:
        current_ip = await fetch_public_ip()
        return {
            "changed": current_ip is not None,
            "current_ip": current_ip,
            "elapsed_seconds": 0.0,
        }

    started_at = asyncio.get_running_loop().time()
    while True:
        current_ip = await fetch_public_ip()
        elapsed_seconds = asyncio.get_running_loop().time() - started_at
        if current_ip and current_ip != previous_ip:
            return {
                "changed": True,
                "current_ip": current_ip,
                "elapsed_seconds": round(elapsed_seconds, 2),
            }
        if elapsed_seconds >= VPN_IP_CHANGE_TIMEOUT_SECONDS:
            return {
                "changed": False,
                "current_ip": current_ip,
                "elapsed_seconds": round(elapsed_seconds, 2),
            }
        await asyncio.sleep(VPN_IP_POLL_INTERVAL_SECONDS)


In [ ]:
VPN_COUNTRY_SEQUENCE


In [ ]:
before_ip = await fetch_public_ip()
target_country, vpn_country_index = get_next_vpn_country(vpn_country_index)
vpn_test_result = {
    "before_ip": before_ip,
    "target_country": target_country,
}

vpn_result = await switch_vpn_server(target_country)
vpn_test_result["vpn_result"] = vpn_result

if vpn_result["success"]:
    print("공인 IP 변경 여부를 확인합니다.")
    ip_change_result = await wait_for_public_ip_change(before_ip)
    vpn_test_result["ip_change_result"] = ip_change_result
    vpn_test_result["after_ip"] = ip_change_result["current_ip"]
else:
    vpn_test_result["ip_change_result"] = None
    vpn_test_result["after_ip"] = None

print(json.dumps(vpn_test_result, ensure_ascii=False, indent=2))
